### Environment setup and data mapping
In this initial phase, the fundamental libraries for tabular data manipulation (Pandas, NumPy), visualization (Matplotlib, Seaborn), and Computer Vision processing (Pillow, OpenCV) are imported. 
Additionally, the path to the directory containing the raw images is defined, and a dictionary for manual labeling is initialized. This dictionary associates each file with its respective artistic movement, providing the categorical baseline necessary to evaluate the performance of segmentation models in relation to painting styles.

In [4]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageStat
import cv2

In [5]:
data_dir = "../data/raw_images/"

artistic_movement = {
    "img_01.jpg": "Post-Impressionism",
    "img_02.jpeg": "Impressionism",
    "img_03.jpg": "Impressionism", 
    "img_04.jpg": "Romanticism",
    "img_05.jpg": "Baroque", 
    "img_06.jpg": "Renaissance",
    "img_07.jpg": "Expressionism",
    "img_08.jpg": "Impressionism",
    "img_09.jpeg": "Post-Impressionism",
    "img_10.png": "Realism", 
    "img_11.jpg": "Realism", 
    "img_12.jpg": "Impressionism",
    "img_13.jpg": "Renaissance", 
    "img_14.jpg": "Vedutism",
    "img_15.jpg": "Renaissance"
}

images_data = []
images_cache = {}

### Visual and mathematical feature extraction
This block constitutes the core of the feature extraction process. By iterating over the entire image dataset, the algorithm extracts and calculates specific metrics for each artwork:
* **Geometric Properties:** Width, height, and aspect ratio.
* **Chromatic Statistics (RGB):** Mean and standard deviation for the red, green, and blue channels, which are essential for quantifying color dominance and channel-specific contrast.
* **Luminance and Structural Complexity:** Following a grayscale conversion, the mean brightness, global contrast, and image entropy are calculated. Entropy serves as an indicator of texture complexity and visual chaos.
* **Edge Intensity (Edge Detection):** By applying the Sobel filter via OpenCV, the magnitude of the gradients on both axes is calculated to obtain a mean value of edge intensity.
The extracted data dynamically populates a list of dictionaries, structured for the subsequent conversion into a Pandas DataFrame.

In [6]:
for filename in sorted(os.listdir(data_dir)):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        filepath = os.path.join(data_dir, filename)
        try:
            with Image.open(filepath) as img:
                img_rgb = img.convert('RGB')
                width, height = img_rgb.size
                aspect_ratio = width/height

                #RGB channels' means
                stats = ImageStat.Stat(img_rgb)
                mean_r, mean_g, mean_b = stats.mean

                #standard deviation for each channel (contrast)
                dev_r, dev_g, dev_b = stats.stddev
                
                #Black and white images have a single channel, so we extract the firt element of the list
                img_gray = img.convert('L')
                gray_stats = ImageStat.Stat(img_gray)
                brightness = gray_stats.mean[0]

                global_contrast = gray_stats.stddev[0]

                entropy = img_gray.entropy()


                #use of Sobel filter 
                img_cv_gray = np.array(img_gray)
                sobel_x = cv2.Sobel(img_cv_gray, cv2.CV_64F, 1, 0, ksize=3)
                sobel_y = cv2.Sobel(img_cv_gray, cv2.CV_64F, 0, 1, ksize=3)
                magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
                mean_edge_intensity = np.mean(magnitude)


                movement = artistic_movement.get(filename)

                images_cache[filename] = {
                    'rgb': np.array(img_rgb),
                    'sobel': magnitude
                }

                images_data.append({
                    'Filename': filename, 
                    'Artistic_movement': movement, 
                    'Width': width, 
                    'Height': height, 
                    'Aspect_ratio': aspect_ratio, 
                    'Mean_brightness': brightness,
                    'Global_contrast': global_contrast, 
                    'Entropy': entropy, 
                    'Edge_intensity_mean': mean_edge_intensity, 
                    'Mean_R': mean_r, 
                    'Mean_G': mean_g, 
                    'Mean_B': mean_b, 
                    'Dev_R': dev_r, 
                    'Dev_G': dev_g, 
                    'Dev_B': dev_b
                })


        except Exception as e:
            print(f"Errore dell'elaborazione di {filename}: {e}")

### DataFrame construction and visual comparison
The aggregated list of dictionaries is converted into a structured Pandas DataFrame. 

Subsequently, a side-by-side visual comparison is generated for each artwork. By retrieving the pre-computed RGB matrices and Sobel magnitude arrays directly from the memory cache, the algorithm plots the original image against its corresponding **Sobel Edge Map**. This visual inspection allows for an immediate qualitative assessment of the structural complexity and edge intensity that were numerically quantified in the previous steps.

In [ ]:
df_images = pd.DataFrame(images_data)
display(df_images)

In [ ]:
for index, row in df_images.iterrows():
    filename = row['Filename']

    img_rgb = images_cache[filename]['rgb']
    magnitude = images_cache[filename]['sobel']

    fig, axes = plt.subplots(1, 2, figsize = (8, 3))

    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Original image", fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(magnitude, cmap='gray')
    axes[1].set_title(f"Sobel edge map\n Edge intensity mean: {row['Edge_intensity_mean']:.2f} ", fontsize=10)
    axes[1].axis('off')

    plt.tight_layout()

In [ ]:
first_file = df_images['Filename'].iloc[0]
img_rgb = images_cache[first_file]['rgb']

plt.figure(figsize=(10, 4))
colors = ('red', 'green', 'blue')

for i, color in enumerate(colors):
    histogram = cv2.calcHist([img_rgb], [i], None, [256], [0, 256])
    plt.plot(histogram, color = color, alpha = 0.8, linewidth = 2)
    plt.fill_between(range(256), histogram.flatten(), color = colors, alpha = 0.1)

plt.xlim([0, 256])